# RUNECLAW V4 — Full-Spectrum Trading Intelligence

Train RUNECLAW V4 on the **76K+ merged dataset** (V3 base + Claude enrichment + V4 new categories).

**What's new in V4:**
- **76,343 training samples** (up from ~54K in V3)
- 5 new training categories:
  1. **Confidence Calibration** (4K) — maps confidence scores to realistic win rates
  2. **Multi-Timeframe Reasoning** (3K) — joint 1H+4H+1D analysis with HTF override
  3. **Risk-Aware Thesis** (3K) — generates trades within pre-specified risk constraints
  4. **Order Flow Intelligence** (3K) — smart-money signals (CVD, book imbalance, whale bias, funding)
  5. **Trade Outcome Feedback** (3K) — post-mortem analysis with lessons learned
- Updated system prompt with order flow, multi-TF, and calibrated confidence capabilities
- Cosine schedule with warmup for smoother convergence on larger dataset
- Model name: `runeclaw-v4`

**Setup:** Runtime → Change runtime type → **L4 GPU** (or A100 if available)

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install safetensors sentencepiece protobuf
print("Dependencies installed.")

## 2. GPU Detection & Hyperparameter Selection

Auto-selects the best configuration for your GPU. Larger dataset benefits from slightly lower LR.

In [ ]:
import torch
import os
import json
import random
import time
import glob

random.seed(42)

gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1024**3

print(f"GPU:  {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

# ── Auto-select best config for available GPU ──
# V4: slightly lower LR for 76K dataset (more steps = needs less aggressive LR)
if vram_gb >= 70:
    MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
    LORA_RANK = 128
    LORA_ALPHA = 128
    BATCH_SIZE = 8
    GRAD_ACCUM = 2
    MAX_SEQ = 2048
    NUM_EPOCHS = 3
    LEARNING_RATE = 3e-5
    tier = "A100-80GB MAX"
elif vram_gb >= 35:
    MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
    LORA_RANK = 128
    LORA_ALPHA = 128
    BATCH_SIZE = 4
    GRAD_ACCUM = 4
    MAX_SEQ = 2048
    NUM_EPOCHS = 3
    LEARNING_RATE = 3e-5
    tier = "A100-40GB"
elif vram_gb >= 20:
    MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
    LORA_RANK = 64
    LORA_ALPHA = 64
    BATCH_SIZE = 2
    GRAD_ACCUM = 8
    MAX_SEQ = 2048
    NUM_EPOCHS = 3
    LEARNING_RATE = 4e-5
    tier = "L4/A10"
elif vram_gb >= 14:
    MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
    LORA_RANK = 32
    LORA_ALPHA = 32
    BATCH_SIZE = 2
    GRAD_ACCUM = 8
    MAX_SEQ = 1024
    NUM_EPOCHS = 3
    LEARNING_RATE = 4e-5
    tier = "T4"
else:
    MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
    LORA_RANK = 16
    LORA_ALPHA = 16
    BATCH_SIZE = 2
    GRAD_ACCUM = 4
    MAX_SEQ = 512
    NUM_EPOCHS = 3
    LEARNING_RATE = 5e-5
    tier = "Low VRAM"

WARMUP_RATIO = 0.03  # V4: lower warmup ratio — 76K samples means plenty of steps
MODEL_NAME = "runeclaw-v4"

# V4 system prompt — expanded with order flow, multi-TF, calibrated confidence
SYSTEM_PROMPT = (
    "You are RUNECLAW V4, an AI trading analyst with full-spectrum market intelligence. "
    "You analyze cryptocurrency markets using the GetClaw Confluence Engine (12 weighted "
    "indicators: RSI-14 w=1.5, MACD w=1.0, Bollinger w=1.2, EMA Cross w=1.0, Volume "
    "Profile w=1.3, OBV w=0.8, Stoch RSI w=0.7, ADX w=1.1, Ichimoku w=0.9, VWAP w=1.4, "
    "ATR w=0.8, Fibonacci w=1.0), enforce strict risk management through 22 automated "
    "checks (all must pass, fail-closed), and generate structured trade ideas. "
    "V4 capabilities: (1) Calibrated confidence — confidence scores map to expected win "
    "rates with realistic outcome distributions. (2) Multi-timeframe reasoning — joint "
    "1H+4H+1D analysis with higher-timeframe override when signals conflict. "
    "(3) Risk-aware thesis — position sizing respects portfolio constraints, max loss, "
    "exposure limits, and regime multipliers. (4) Order flow intelligence — interprets "
    "book imbalance, CVD, whale activity, funding rates, OI-price divergence, and "
    "spot-futures divergence for smart money positioning. (5) Trade outcome feedback — "
    "learns from wins and losses with structured post-mortem analysis. "
    "You never execute without human confirmation. Capital preservation above all."
)

eff_batch = BATCH_SIZE * GRAD_ACCUM
print(f"\n{'='*50}")
print(f"  Config: {tier}")
print(f"{'='*50}")
print(f"  Model:     {MODEL}")
print(f"  LoRA:      rank={LORA_RANK}, alpha={LORA_ALPHA}")
print(f"  Context:   {MAX_SEQ} tokens")
print(f"  Batch:     {BATCH_SIZE} x {GRAD_ACCUM} = {eff_batch} effective")
print(f"  Epochs:    {NUM_EPOCHS}")
print(f"  LR:        {LEARNING_RATE}")
print(f"  Warmup:    {WARMUP_RATIO*100:.0f}% of steps")

## 3. Load Training Data

**Upload your `combined_training_v4.jsonl`** (~76K samples: V3 + Claude + V4 new categories)

Created by running `python generate_v4_training_data.py` locally. If you don't have the file, synthetic data will be generated as fallback.

In [ ]:
# ── Upload your merged V4 training data ──
from google.colab import files

DATA_FILE = "/content/training_data.jsonl"
all_samples = []

print("Upload your combined_training_v4.jsonl (~76K samples):")
print("(V3 base + Claude enrichment + V4 new categories)")
print("(Click Cancel to generate synthetic data instead)\n")
try:
    uploaded = files.upload()
    for fname, content in uploaded.items():
        lines = content.decode('utf-8').strip().split('\n')
        for line in lines:
            line = line.strip()
            if line:
                try:
                    sample = json.loads(line)
                    if 'instruction' in sample and 'output' in sample:
                        all_samples.append(sample)
                except json.JSONDecodeError:
                    pass
        print(f"Loaded {len(all_samples):,} samples from {fname}")
except Exception as e:
    print(f"No file uploaded. Will generate synthetic data in next cell.")

if all_samples:
    print(f"\nDataset ready: {len(all_samples):,} samples")
    # Quality stats
    approved = sum(1 for s in all_samples if 'APPROVED' in s.get('output','').upper())
    rejected = sum(1 for s in all_samples if 'REJECTED' in s.get('output','').upper())
    json_fmt = sum(1 for s in all_samples if s.get('output','').strip().startswith('{'))
    has_input = sum(1 for s in all_samples if s.get('input','').strip())
    avg_len = sum(len(s.get('output','')) for s in all_samples) / len(all_samples)

    # V4 category detection
    confidence_cal = sum(1 for s in all_samples if 'CONFIDENCE ASSESSMENT' in s.get('output',''))
    multi_tf = sum(1 for s in all_samples if 'MULTI-TIMEFRAME ANALYSIS' in s.get('output',''))
    risk_aware = sum(1 for s in all_samples if 'RISK-AWARE TRADE' in s.get('output',''))
    order_flow = sum(1 for s in all_samples if 'ORDER FLOW ANALYSIS' in s.get('output',''))
    trade_feedback = sum(1 for s in all_samples if 'TRADE REFLECTION' in s.get('output',''))

    print(f"  Approved:    {approved:,}")
    print(f"  Rejected:    {rejected:,}")
    print(f"  JSON format: {json_fmt:,}")
    print(f"  Has input:   {has_input:,}")
    print(f"  Avg output:  {avg_len:.0f} chars")
    print(f"\n  V4 Categories:")
    print(f"    Confidence Calibration: {confidence_cal:,}")
    print(f"    Multi-Timeframe:        {multi_tf:,}")
    print(f"    Risk-Aware Thesis:      {risk_aware:,}")
    print(f"    Order Flow:             {order_flow:,}")
    print(f"    Trade Feedback:          {trade_feedback:,}")
    GENERATE_SYNTHETIC = False
else:
    GENERATE_SYNTHETIC = True
    print("Will generate synthetic data in next cell.")

In [ ]:
# ── Synthetic data fallback (only runs if no file was uploaded) ──
if GENERATE_SYNTHETIC:
    print("Generating synthetic training data (V4 categories included)...")

    PAIRS = [
        "BTC/USDT", "ETH/USDT", "SOL/USDT", "BNB/USDT", "XRP/USDT",
        "ADA/USDT", "AVAX/USDT", "DOGE/USDT", "DOT/USDT", "MATIC/USDT",
        "LINK/USDT", "UNI/USDT", "ATOM/USDT", "FIL/USDT", "APT/USDT",
        "ARB/USDT", "OP/USDT", "SUI/USDT", "SEI/USDT", "INJ/USDT",
        "TIA/USDT", "JUP/USDT", "WIF/USDT", "NEAR/USDT", "RENDER/USDT",
        "FET/USDT", "RUNE/USDT", "STX/USDT", "ICP/USDT", "PEPE/USDT",
    ]
    INDICATORS = [
        "RSI", "MACD", "Bollinger Bands", "EMA Cross", "Volume Profile",
        "OBV", "Stochastic RSI", "ADX", "Ichimoku Cloud", "VWAP",
        "Fibonacci Retracement", "ATR",
    ]
    RISK_CHECKS = [
        "max_position_size", "portfolio_heat", "correlation_limit",
        "drawdown_threshold", "volatility_filter", "liquidity_check",
        "spread_limit", "funding_rate", "open_interest_divergence",
        "whale_activity", "exchange_reserve", "news_sentiment",
        "time_of_day", "consecutive_losses", "win_rate_threshold",
        "max_leverage", "stop_loss_required", "take_profit_required",
        "risk_reward_minimum", "daily_loss_limit", "weekly_loss_limit",
        "margin_utilization", "circuit_breaker",
    ]
    REGIMES = ["TRENDING_UP", "TRENDING_DOWN", "RANGING", "VOLATILE",
               "CHOPPY", "ACCUMULATION", "DISTRIBUTION", "STRONG_TREND"]
    REGIME_MULTIPLIERS = {
        "CHOPPY": 0.5, "RANGING": 0.7, "VOLATILE": 0.3,
        "TRENDING_UP": 1.0, "TRENDING_DOWN": 1.0,
        "STRONG_TREND": 1.5, "ACCUMULATION": 0.8, "DISTRIBUTION": 0.6,
    }
    PRICES = {
        "BTC": 105000, "ETH": 2750, "SOL": 178, "BNB": 720, "XRP": 2.55,
        "ADA": 0.78, "AVAX": 25, "DOGE": 0.088, "DOT": 4.8, "MATIC": 0.25,
        "LINK": 16, "UNI": 7.5, "ATOM": 5.2, "FIL": 3.3, "APT": 6.1,
        "ARB": 0.45, "OP": 0.95, "SUI": 4.2, "SEI": 0.28, "INJ": 14,
        "TIA": 3.8, "JUP": 0.62, "WIF": 1.1, "PEPE": 0.000014, "RENDER": 5.5,
        "FET": 1.2, "NEAR": 3.8, "ICP": 6.5, "RUNE": 2.1, "STX": 0.85,
    }

    def rp(base, pct=0.1):
        raw = base * (1 + random.uniform(-pct, pct))
        if base < 0.01: return round(raw, 8)
        elif base < 1: return round(raw, 6)
        elif base < 100: return round(raw, 4)
        return round(raw, 2)

    def rpct(lo, hi): return round(random.uniform(lo, hi), 2)
    def gpp(pair): return PRICES.get(pair.split('/')[0], 10.0)

    # 1. Trade Analysis (2000)
    for _ in range(2000):
        pair = random.choice(PAIRS); tf = random.choice(["15m","1h","4h","1d"])
        d = random.choice(["long", "short"]); regime = random.choice(REGIMES)
        mult = REGIME_MULTIPLIERS[regime]; base = gpp(pair); entry = rp(base, 0.02)
        if entry == 0: entry = base
        base_risk = rpct(1.5, 3.0); final_risk = round(base_risk * mult, 2)
        if d == "long":
            stop = round(entry * (1 - random.uniform(0.01, 0.04)), 6)
            tp1 = round(entry * (1 + random.uniform(0.02, 0.06)), 6)
        else:
            stop = round(entry * (1 + random.uniform(0.01, 0.04)), 6)
            tp1 = round(entry * (1 - random.uniform(0.02, 0.06)), 6)
        risk_pct = round(abs(entry - stop) / entry * 100, 2) if entry > 0 else 1.0
        reward_pct = round(abs(tp1 - entry) / entry * 100, 2) if entry > 0 else 1.0
        rr = round(reward_pct / risk_pct, 2) if risk_pct > 0 else 0
        conf = rpct(0.45, 0.95); ni = random.randint(6, 12)
        bl = random.randint(3, ni); brl = ni - bl
        ai = random.sample(INDICATORS, min(ni, len(INDICATORS)))
        out = (f"Trade Idea: {pair}\nDirection: {d.upper()}\nTimeframe: {tf}\n"
               f"Market Regime: {regime}\n\nEntry: {entry}\nStop Loss: {stop} ({risk_pct}%)\n"
               f"Take Profit 1: {tp1} ({reward_pct}%)\nRisk:Reward: 1:{rr}\n\n"
               f"Confluence: {conf} ({ni} indicators)\n"
               f"Bullish ({bl}): {', '.join(ai[:bl])}\nBearish ({brl}): {', '.join(ai[bl:])}\n\n"
               f"Position Size: {final_risk}% of portfolio\n"
               f"  Base risk: {base_risk}% x regime multiplier {mult}x ({regime}) = {final_risk}%\n"
               f"Status: {'APPROVED' if conf > 0.6 and rr >= 1.5 else 'REJECTED'}")
        all_samples.append({"instruction": f"Analyze {pair} on {tf} for trade setups.", "input": "", "output": out})

    # 2. JSON format trades (1500)
    for _ in range(1500):
        pair = random.choice(PAIRS); tf = random.choice(["1h","4h","1d"])
        d = random.choice(["LONG", "SHORT"]); regime = random.choice(REGIMES)
        base = gpp(pair); entry = rp(base, 0.02)
        if entry == 0: entry = base
        if d == "LONG":
            stop = round(entry * (1 - random.uniform(0.01, 0.04)), 6)
            tp = round(entry * (1 + random.uniform(0.02, 0.06)), 6)
        else:
            stop = round(entry * (1 + random.uniform(0.01, 0.04)), 6)
            tp = round(entry * (1 - random.uniform(0.02, 0.06)), 6)
        conf = rpct(0.50, 0.95)
        rr_val = round(abs(tp - entry) / abs(entry - stop), 2) if abs(entry - stop) > 0 else 0
        signals = random.sample(INDICATORS, random.randint(5, 10))
        reasoning = random.choice([
            f"{pair} showing strong {d.lower()} momentum with ADX > 25 and MACD crossover.",
            f"Volume surge with OBV confirmation on {tf}. {regime} regime favors {d.lower()} entries.",
            f"EMA 20/50 aligned {d.lower()}, RSI at {random.randint(30,70)}, BB squeeze breakout.",
        ])
        json_out = json.dumps({"direction": d, "confidence": conf, "entry_price": entry,
            "stop_loss": stop, "take_profit": tp, "risk_reward": rr_val, "regime": regime,
            "signals_used": signals, "reasoning": reasoning,
            "verdict": "APPROVED" if conf > 0.6 and rr_val >= 1.5 else "REJECTED"}, indent=2)
        all_samples.append({"instruction": f"Analyze {pair} on {tf}. Return JSON.", "input": "", "output": json_out})

    # 3. No-Trade Rejections (2000)
    for _ in range(2000):
        pair = random.choice(PAIRS); d = random.choice(["long","short"])
        reason = random.choice(["Low confluence", "R:R below minimum", "Portfolio heat exceeded",
            "Consecutive losses", "Daily drawdown limit", "Crowded funding rate", "Circuit breaker active"])
        nf = random.randint(1, 5); fc = random.sample(RISK_CHECKS, nf)
        out = (f"Trade Scan: {pair} ({d.upper()})\nStatus: REJECTED\n\nReason: {reason}\n\n"
               f"Failed checks ({nf}/{len(RISK_CHECKS)}):\n")
        for c in fc: out += f"  FAIL: {c}\n"
        out += "\nAction: NO TRADE. Capital preservation above all."
        all_samples.append({"instruction": f"Scan {pair} for {d} setups.", "input": "", "output": out})

    # 4. Confidence Calibration (2000)
    bands = [(0.50,0.55,0.48,0.55),(0.55,0.60,0.52,0.60),(0.60,0.65,0.58,0.66),
             (0.65,0.70,0.62,0.72),(0.70,0.75,0.68,0.78),(0.75,0.80,0.72,0.82),(0.80,0.90,0.75,0.88)]
    for _ in range(2000):
        pair = random.choice(PAIRS); base = gpp(pair); entry = rp(base, 0.03)
        if entry == 0: entry = base
        direction = random.choice(["LONG", "SHORT"]); regime = random.choice(REGIMES)
        band = random.choice(bands); raw_conf = rpct(band[0], band[1]); actual_wr = rpct(band[2], band[3])
        rsi = rpct(25, 75); adx = rpct(12, 55)
        sl_pct = rpct(1.5, 4.0); rr = rpct(1.2, 3.5); tp_pct = round(sl_pct * rr, 2)
        is_win = random.random() < actual_wr; outcome = "WIN" if is_win else "LOSS"
        pnl_pct = rpct(0.5, tp_pct) if is_win else rpct(0.5, sl_pct)
        out = (f"CONFIDENCE ASSESSMENT: {pair} {direction}\n{'='*45}\n\n"
               f"Confluence Score: {raw_conf:.2f}\nCalibrated Confidence: {raw_conf:.0%}\n\n"
               f"Expected win rate at {raw_conf:.0%} confidence: ~{actual_wr:.0%}\n"
               f"Expected value: {(actual_wr * tp_pct - (1-actual_wr) * sl_pct):+.2f}% per trade\n\n"
               f"Actual Result: {outcome} ({'+' if is_win else '-'}{pnl_pct}%)\n\n"
               f"Verdict: {'APPROVED' if raw_conf > 0.60 and rr >= 1.2 else 'REJECTED'}")
        all_samples.append({"instruction": f"Assess trade confidence for {pair} {direction}.",
                           "input": f"RSI: {rsi} | ADX: {adx} | R:R: 1:{rr}", "output": out})

    # 5. Order Flow Intelligence (1500)
    for _ in range(1500):
        pair = random.choice(PAIRS); base = gpp(pair); entry = rp(base, 0.03)
        if entry == 0: entry = base
        book_imb = rpct(-0.8, 0.8); cvd = random.choice(["rising","falling","flat"])
        whale = random.choice(["accumulation","distribution","neutral"])
        funding = round(random.uniform(-0.03, 0.08), 4)
        oi_change = rpct(-15, 20); taker = rpct(0.3, 1.8)
        bull = sum([book_imb > 0.2, cvd == "rising", whale == "accumulation",
                    funding < -0.01, taker > 1.1])
        bear = sum([book_imb < -0.2, cvd == "falling", whale == "distribution",
                    funding > 0.03, taker < 0.8])
        bias = "BULLISH" if bull > bear + 1 else "BEARISH" if bear > bull + 1 else "NEUTRAL"
        out = (f"ORDER FLOW ANALYSIS: {pair}\n{'='*45}\n\n"
               f"Book imbalance: {book_imb:+.2f} | CVD: {cvd} | Whale: {whale}\n"
               f"Funding: {funding:+.4f} | OI change: {oi_change:+.1f}% | Taker: {taker:.2f}\n\n"
               f"Overall Bias: {bias} (bull signals: {bull}, bear signals: {bear})")
        all_samples.append({"instruction": f"Analyze order flow for {pair}.", "input": "", "output": out})

    # 6. Trade Outcome Feedback (1500)
    for _ in range(1500):
        pair = random.choice(PAIRS); base = gpp(pair); entry = rp(base, 0.03)
        if entry == 0: entry = base
        direction = random.choice(["LONG","SHORT"])
        outcome = random.choice(["TP_HIT","SL_HIT","PARTIAL_WIN","BREAKEVEN"])
        pnl = rpct(-4, 6) if outcome != "BREAKEVEN" else 0.0
        if outcome == "SL_HIT": pnl = -abs(pnl)
        elif outcome in ("TP_HIT","PARTIAL_WIN"): pnl = abs(pnl)
        lesson = random.choice([
            "Regime correctly identified", "SL placement gave adequate room",
            "Counter-trend entry — should have reduced size", "SL too tight",
            "Volume declining during entry — weak momentum", "HTF was counter-directional",
        ])
        out = (f"TRADE REFLECTION: {pair} {direction}\n{'='*45}\n\n"
               f"Outcome: {outcome} ({pnl:+.2f}%)\n"
               f"Result: {'WIN' if pnl > 0 else 'LOSS' if pnl < 0 else 'BREAKEVEN'}\n\n"
               f"Lesson: {lesson}")
        all_samples.append({"instruction": f"Reflect on this {pair} {direction} trade.",
                           "input": f"Outcome: {outcome}, PnL: {pnl:+.2f}%", "output": out})

    random.shuffle(all_samples)
    print(f"\nGenerated {len(all_samples):,} synthetic samples (V3 + V4 categories)")
else:
    print(f"Using uploaded data: {len(all_samples):,} samples (skipped synthetic generation)")

## 4. Load Model & Apply LoRA

In [ ]:
from unsloth import FastLanguageModel

print(f"Loading {MODEL}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    dtype=None,
    load_in_4bit=True,
)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Applying LoRA (rank={LORA_RANK})...")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

## 5. Tokenize Dataset

In [ ]:
# ── Format all samples into chat template ──
print(f"Formatting {len(all_samples):,} samples...")

formatted_texts = []
for ex in all_samples:
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    user_msg = ex["instruction"]
    if ex.get("input", "").strip():
        user_msg += "\n\n" + ex["input"]
    msgs.append({"role": "user", "content": user_msg})
    msgs.append({"role": "assistant", "content": ex["output"]})
    formatted_texts.append(
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    )

# Spot check token lengths
check_n = min(500, len(formatted_texts))
lens = [len(tokenizer(t, truncation=False)["input_ids"]) for t in formatted_texts[:check_n]]
print(f"Token lengths (sample of {check_n}):")
print(f"  Avg: {sum(lens)/len(lens):.0f}, Max: {max(lens)}, Min: {min(lens)}")
print(f"  Under {MAX_SEQ}: {sum(1 for l in lens if l <= MAX_SEQ)}/{check_n}")

In [ ]:
# ── Tokenize into training dataset ──
from torch.utils.data import Dataset as TorchDataset
from torch.nn.utils.rnn import pad_sequence

class TrainDataset(TorchDataset):
    def __init__(self, texts, tok, max_len):
        self.items = []
        print(f"  Tokenizing {len(texts):,} samples...")
        for i, t in enumerate(texts):
            enc = tok(t, truncation=True, max_length=max_len, padding=False, return_tensors="pt")
            ids = enc["input_ids"].squeeze()
            self.items.append({
                "input_ids": ids,
                "attention_mask": torch.ones_like(ids),
                "labels": ids.clone(),
            })
            if (i + 1) % 10000 == 0:
                print(f"    {i+1:,}/{len(texts):,}...")
        print(f"  Done: {len(self.items):,} samples tokenized.")

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

class PadCollator:
    def __init__(self, pad_id):
        self.pad_id = pad_id
    def __call__(self, batch):
        return {
            "input_ids": pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=self.pad_id),
            "labels": pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=-100),
            "attention_mask": pad_sequence([b["attention_mask"] for b in batch], batch_first=True, padding_value=0),
        }

train_dataset = TrainDataset(formatted_texts, tokenizer, MAX_SEQ)
collator = PadCollator(tokenizer.pad_token_id)
print(f"\nDataset ready: {len(train_dataset):,} samples")

## 6. Train

V4 training on 76K samples. On L4 GPU with batch 16 effective, expect ~14K steps per epoch.

In [ ]:
from transformers import Trainer, TrainingArguments

total_steps = len(train_dataset) * NUM_EPOCHS // (BATCH_SIZE * GRAD_ACCUM)
warmup_steps = int(total_steps * WARMUP_RATIO)

print(f"{'='*50}")
print(f"  TRAINING CONFIG — RUNECLAW V4")
print(f"{'='*50}")
print(f"  Samples:     {len(train_dataset):,}")
print(f"  Epochs:      {NUM_EPOCHS}")
print(f"  Total steps: {total_steps:,}")
print(f"  Warmup:      {warmup_steps} steps ({WARMUP_RATIO*100:.0f}%)")
print(f"  Batch:       {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LoRA rank:   {LORA_RANK}")
print(f"  LR:          {LEARNING_RATE}")
print()

training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=warmup_steps,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    output_dir=f"/content/{MODEL_NAME}-checkpoints",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    dataloader_pin_memory=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
)

print("Starting training...")
stats = trainer.train()

print(f"\n{'='*50}")
print(f"  TRAINING COMPLETE — RUNECLAW V4")
print(f"{'='*50}")
print(f"  Final loss:  {stats.training_loss:.4f}")
print(f"  Runtime:     {stats.metrics['train_runtime']:.0f}s ({stats.metrics['train_runtime']/3600:.1f}h)")
print(f"  Samples/sec: {stats.metrics.get('train_samples_per_second', 0):.1f}")

## 7. Quick Test

Verify the model produces sensible output across all V4 categories before exporting.

In [ ]:
FastLanguageModel.for_inference(model)

# V4 test prompts — covering all new categories + V3 fundamentals
test_prompts = [
    # V3 fundamentals
    "Analyze BTC/USDT on 4h for trade setups.",
    "Analyze ETH/USDT at $2750 on 1h and output a structured JSON trade decision.",
    "Calculate position size for SOL/USDT with $25,000 portfolio in VOLATILE regime.",
    # V4: Confidence Calibration
    "Assess trade confidence for DOGE/USDT LONG. Confluence is 0.72 with 9/12 indicators bullish, ADX 32, RSI 55. What's the expected win rate?",
    # V4: Multi-Timeframe
    "Multi-timeframe analysis for SOL/USDT: 1H bullish (RSI 58, MACD cross up), 4H bullish (above EMA 50), 1D bearish (RSI 68, near resistance). What's the alignment verdict?",
    # V4: Risk-Aware Thesis
    "I have $10,000 portfolio, max 2% loss per trade, max 3 positions (currently 2 open). Can I take a LONG on ARB/USDT at $0.45 with 3% SL in TRENDING_UP regime?",
    # V4: Order Flow Intelligence
    "Analyze order flow for BTC/USDT: book imbalance +0.45, CVD rising, whale accumulation detected, funding rate +0.06%, OI up 12%, taker ratio 1.3. What's the smart money signal?",
    # V4: Trade Outcome Feedback
    "Reflect on this ETH/USDT SHORT trade: entered at $2800, SL $2884 (3%), TP $2660 (5%), hit SL after 4 hours. Confidence was 0.58. What went wrong?",
]

for prompt_text in test_prompts:
    test_prompt = tokenizer.apply_chat_template([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text},
    ], tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=512,
            temperature=0.3, top_p=0.9,
            do_sample=True,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt_text[:80]}...")
    print(f"{'='*60}")
    print(response[:800])
    print()

## 8. Export to GGUF

Export Q4_K_M (fast, ~5GB) and Q8_0 (higher quality, ~8GB — fits RTX 4080).

In [ ]:
OUTPUT_DIR_GGUF = f"/content/{MODEL_NAME}-gguf"

# ── Export Q4_K_M (fast inference, smaller file) ──
print("Exporting Q4_K_M quantization...")
model.save_pretrained_gguf(
    OUTPUT_DIR_GGUF,
    tokenizer,
    quantization_method="q4_k_m",
)

print(f"\nQ4_K_M output directory:")
for fn in sorted(os.listdir(OUTPUT_DIR_GGUF)):
    src = f"{OUTPUT_DIR_GGUF}/{fn}"
    if os.path.isdir(src):
        print(f"  [DIR] {fn}/")
    else:
        size = os.path.getsize(src)
        if size > 1024**2:
            print(f"  {fn}: {size/1024**3:.2f} GB")
        else:
            print(f"  {fn}: {size/1024:.1f} KB")

# ── Export Q8_0 (higher quality, fits RTX 4080 16GB) ──
OUTPUT_DIR_Q8 = f"/content/{MODEL_NAME}-gguf-q8"
print("\nExporting Q8_0 quantization (higher quality)...")
try:
    model.save_pretrained_gguf(
        OUTPUT_DIR_Q8,
        tokenizer,
        quantization_method="q8_0",
    )
    has_q8 = True
except Exception as e:
    print(f"  Q8 export failed (may need more RAM): {e}")
    has_q8 = False

# Find all GGUF files
gguf_files = glob.glob(f"{OUTPUT_DIR_GGUF}/**/*.gguf", recursive=True)
if not gguf_files:
    gguf_files = [f"{OUTPUT_DIR_GGUF}/{fn}" for fn in os.listdir(OUTPUT_DIR_GGUF)
                  if os.path.isfile(f"{OUTPUT_DIR_GGUF}/{fn}") and os.path.getsize(f"{OUTPUT_DIR_GGUF}/{fn}") > 1024**3]

gguf_files_q8 = []
if has_q8:
    gguf_files_q8 = glob.glob(f"{OUTPUT_DIR_Q8}/**/*.gguf", recursive=True)
    if not gguf_files_q8:
        gguf_files_q8 = [f"{OUTPUT_DIR_Q8}/{fn}" for fn in os.listdir(OUTPUT_DIR_Q8)
                         if os.path.isfile(f"{OUTPUT_DIR_Q8}/{fn}") and os.path.getsize(f"{OUTPUT_DIR_Q8}/{fn}") > 1024**3]

print(f"\nGGUF files found:")
for f in gguf_files + gguf_files_q8:
    size = os.path.getsize(f) / 1024**3
    print(f"  {os.path.basename(f)}: {size:.2f} GB")

if not gguf_files:
    print("\nWARNING: No GGUF files found! Check export output above for errors.")

In [ ]:
# ── Create Modelfiles for both quantizations ──
for label, gdir, gfiles in [("Q4_K_M", OUTPUT_DIR_GGUF, gguf_files),
                              ("Q8_0", OUTPUT_DIR_Q8, gguf_files_q8)]:
    if not gfiles:
        continue
    gguf_name = os.path.basename(gfiles[0])

    modelfile_content = f"""FROM ./{gguf_name}

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|end|>"

SYSTEM \"\"\"{SYSTEM_PROMPT}\"\"\"
"""
    mf_path = f"{gdir}/Modelfile"
    with open(mf_path, "w") as f:
        f.write(modelfile_content)
    print(f"{label} Modelfile created: {mf_path}")

print(f"\nAll files in {OUTPUT_DIR_GGUF}:")
for fn in sorted(os.listdir(OUTPUT_DIR_GGUF)):
    size = os.path.getsize(f"{OUTPUT_DIR_GGUF}/{fn}")
    if size > 1024**2:
        print(f"  {fn}: {size/1024**3:.2f} GB")
    else:
        print(f"  {fn}: {size/1024:.1f} KB")

## 9. Save & Download

**Option A:** Direct download (if GGUF < 4GB)  
**Option B:** Save to Google Drive (recommended for large files)

In [ ]:
# ── Save the LoRA adapter (smaller, can merge locally) ──
adapter_dir = f"/content/{MODEL_NAME}-adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"LoRA adapter saved to {adapter_dir}")

# Save final dataset to disk
with open(DATA_FILE, "w", encoding="utf-8") as f:
    for s in all_samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")
print(f"Saved {len(all_samples):,} samples to {DATA_FILE}")

In [ ]:
# ── Option A: Direct download (if GGUF < 4GB) ──
from google.colab import files

if gguf_files:
    gguf_path = gguf_files[0]
    gguf_name = os.path.basename(gguf_path)
    gguf_size = os.path.getsize(gguf_path) / 1024**3
    if gguf_size < 4:
        print(f"Downloading {gguf_name} ({gguf_size:.1f} GB)...")
        files.download(gguf_path)
        modelfile_path = f"{OUTPUT_DIR_GGUF}/Modelfile"
        if os.path.exists(modelfile_path):
            files.download(modelfile_path)
    else:
        print(f"GGUF is {gguf_size:.1f} GB — use Google Drive (Option B) instead.")
else:
    print("No GGUF file found. Use Option B (Google Drive).")

In [ ]:
# ── Option B: Save to Google Drive ──
from google.colab import drive
import shutil

drive.mount('/content/drive')

drive_dir = "/content/drive/MyDrive/runeclaw-model-v4"
os.makedirs(drive_dir, exist_ok=True)

# Copy Q4_K_M GGUF + Modelfile
for fn in os.listdir(OUTPUT_DIR_GGUF):
    src = f"{OUTPUT_DIR_GGUF}/{fn}"
    if os.path.isdir(src):
        print(f"  Skipping directory: {fn}")
        continue
    dst = f"{drive_dir}/{fn}"
    print(f"  Copying {fn}...")
    shutil.copy2(src, dst)

# Copy Q8_0 if available
if has_q8:
    drive_dir_q8 = f"{drive_dir}/q8"
    os.makedirs(drive_dir_q8, exist_ok=True)
    for fn in os.listdir(OUTPUT_DIR_Q8):
        src = f"{OUTPUT_DIR_Q8}/{fn}"
        if os.path.isdir(src):
            continue
        dst = f"{drive_dir_q8}/{fn}"
        print(f"  Copying Q8/{fn}...")
        shutil.copy2(src, dst)

# Also copy training data for reference
shutil.copy2(DATA_FILE, f"{drive_dir}/training_data.jsonl")

# Copy adapter
adapter_drive = f"{drive_dir}/adapter"
os.makedirs(adapter_drive, exist_ok=True)
for fn in os.listdir(adapter_dir):
    src = f"{adapter_dir}/{fn}"
    if os.path.isdir(src):
        continue
    shutil.copy2(src, f"{adapter_drive}/{fn}")

print(f"\nSaved to Google Drive: {drive_dir}")
print(f"\n{'='*50}")
print(f"  RUNECLAW V4 DEPLOYMENT")
print(f"{'='*50}")
print(f"  1. Download folder from Google Drive")
print(f"  2. cd into the folder")
print(f"  3. ollama create runeclaw-v4 -f Modelfile")
print(f"  4. ollama run runeclaw-v4 'Scan BTC/USDT for trade setups'")
print(f"\n  For RTX 4080 (higher quality Q8):")
print(f"  cd q8 && ollama create runeclaw-v4-q8 -f Modelfile")
print(f"\n  For bot integration:")
print(f"  Set LLM_PROVIDER=ollama")
print(f"  Set LLM_MODEL=runeclaw-v4")
print(f"  Set LLM_BASE_URL=http://localhost:11434/v1")
print(f"\n  V4 new capabilities to test:")
print(f"  - 'Assess confidence for BTC LONG with 0.72 confluence'")
print(f"  - 'Analyze 1H+4H+1D alignment for SOL/USDT'")
print(f"  - 'Order flow analysis: CVD rising, whale accumulation, funding +0.05%'")
print(f"  - 'Reflect on this losing trade and extract lessons'")